# Customer simulator: large-scale simulation on Kaggle
Runs simulator x agent x scenario x seed on a GPU with the HF/vLLM backends, then evaluates.
The run resumes automatically if the session stops: re-run the simulate cell.

In [ ]:
!pip install -q unsloth vllm  # TODO(verify) versions compatible with Kaggle's CUDA
!git clone https://github.com/TODO-USER/TODO-REPO.git code
%cd code
!pip install -q --no-deps -e .

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

In [ ]:
# Data (same split as training: same config + seed)
!mkdir -p data/raw
!wget -q -O data/raw/abcd_v1.1.json.gz https://github.com/asappresearch/abcd/raw/master/data/abcd_v1.1.json.gz
!gunzip -f data/raw/abcd_v1.1.json.gz
!wget -q -O data/raw/kb.json https://github.com/asappresearch/abcd/raw/master/data/kb.json  # TODO(verify)
!python -m usersim.prepare_data --config configs/data.yaml

In [ ]:
# TODO: write configs/simulate_kaggle.yaml with `type: vllm` / `type: hf` backends
#       (implement HFBackend / VLLMBackend in src/usersim/agents.py first),
#       n_scenarios >= 150, seeds [0, 1, 2], enable ft + broken agents.
!python -m usersim.simulate --config configs/simulate_kaggle.yaml --run-id main

In [ ]:
!python -m usersim.evaluate --config configs/eval.yaml
from IPython.display import Markdown
Markdown(open("outputs/eval/results.md").read())

In [ ]:
# Keep results: copy transcripts + eval to /kaggle/working (Kaggle output) before the session ends
!cp -r outputs /kaggle/working/